# Reproducción de los resultados de la monografía (para el revisor)

**Detección de anomalías en la conciliación de materiales para instalaciones de fibra óptica mediante reglas y aprendizaje automático** — Helen Angulo Laime.

Este cuaderno **no necesita la planilla original ni los PDF del operador**. Usa solo la base anonimizada `Datos_PowerBI.xlsx` (Anexo 1) y reproduce las tablas y figuras del Capítulo 4 y del Anexo 5.

**Cómo usarlo:** menú *Entorno de ejecución → Ejecutar todas*. El cuaderno descarga los archivos del repositorio de GitHub de los anexos. Si no hay internet o el repositorio no está disponible, pedirá subir a mano tres archivos del CD: `Datos_PowerBI.xlsx`, `modelos_v12.py` y `analisis_estadistico.py`.

> El cuaderno `conciliacion_colab.ipynb` es otra cosa: es la herramienta de uso mensual de la empresa y sí requiere la planilla real, que no se publica por confidencialidad.


In [ ]:
# 1. Obtener los archivos del CD (subidos a Colab o del repositorio de GitHub)
import os, glob
NOMBRES = ["Datos_PowerBI.xlsx", "modelos_v12.py", "analisis_estadistico.py"]
REPO = "https://github.com/helen2609/anexos-monografia-conciliacion"
os.makedirs("revision", exist_ok=True)

def buscar(nombre):
    # primero los archivos subidos a la carpeta principal; despues los del repositorio
    if os.path.exists(nombre):
        return nombre
    r = glob.glob(f"anexos/**/{nombre}", recursive=True)
    return r[0] if r else None

if any(buscar(f) is None for f in NOMBRES) and not os.path.isdir("anexos"):
    os.system(f"git clone -q {REPO} anexos")
faltan = [f for f in NOMBRES if buscar(f) is None]
if faltan:
    print("Suba estos archivos del CD (carpetas 1_Datos y 2_Codigo):", faltan)
    from google.colab import files
    files.upload()
    faltan = [f for f in NOMBRES if buscar(f) is None]
if faltan:
    raise FileNotFoundError(f"Siguen faltando {faltan}. Subalos al panel Archivos y vuelva a ejecutar esta celda.")
DATOS, MODELOS, ESTAD = (buscar(f) for f in NOMBRES)
print("Base:", DATOS); print("Modelos:", MODELOS); print("Pruebas de las reglas:", ESTAD)


In [ ]:
# 2. Librerías (scikit-learn, scipy, pandas y matplotlib ya vienen en Colab)
!pip install -q xgboost==3.4.1 openpyxl==3.1.5 scikit-learn==1.9.1


## Pruebas estadísticas de las reglas (apartados 4.2 y 4.6.3)
Intervalos de Wilson, prueba exacta de Fisher y U de Mann-Whitney.

In [ ]:
!python "{ESTAD}" "{DATOS}"


## Catorce enfoques de detección de anomalías (apartado 4.4)
- `MODO_RAPIDO = True`: unos 5 minutos. Todas las cifras son las del documento, salvo el valor *p* de permutación de los tres modelos supervisados, que se deja vacío.
- `MODO_RAPIDO = False`: unos 20 a 30 minutos. Incluye las 200 permutaciones con reentrenamiento de los supervisados (Tabla 4-4 completa).


In [ ]:
MODO_RAPIDO = True
perm = 0 if MODO_RAPIDO else 200
!PERM_SUP_B={perm} python "{MODELOS}" "{DATOS}" revision/


## Tablas principales

In [ ]:
import pandas as pd
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 20)
t = pd.read_csv("revision/Tabla_4-3_comparacion_enfoques.csv")
print("Tabla 4-4 del documento: desempeño de los catorce enfoques")
display(t[["enfoque", "familia", "AUC", "IC95_inf", "IC95_sup", "AP", "AP_tasa_base", "p_permutacion", "p_Holm"]].round(3))
c = pd.read_csv("revision/Tabla_4-4_carga_de_revision.csv")
print("Tabla 4-5 del documento: observadas capturadas según la carga de revisión")
display(c.pivot(index="enfoque", columns="k", values="capturadas_media").round(1).sort_values(119, ascending=False))
print("Tabla 4-3 del documento: perfil de los segmentos")
display(pd.read_csv("revision/Tabla_4-5_perfil_segmentos.csv").round(2))
print("Tabla 4-6 del documento: matriz de confusión con 119 órdenes")
display(pd.read_csv("revision/Tabla_4-6_matriz_confusion_k119.csv"))
print("Tabla A-5: sensibilidad a la definición de la etiqueta")
display(pd.read_csv("revision/Tabla_A-5_sensibilidad_etiqueta.csv").round(3))


## Figuras

In [ ]:
from IPython.display import Image, display
for f, n in [("Figura_4-6_seleccion_grupos.png", "Figura 4-4"), ("Figura_4-7_perfil_grupos.png", "Figura 4-5"),
             ("Figura_4-4_comparacion_enfoques.png", "Figura 4-6"), ("Figura_4-5_curvas_ROC_PR.png", "Figura 4-7"),
             ("Figura_4-8_curva_captura.png", "Figura 4-8"), ("Figura_A-1_proyeccion_PCA.png", "Figura A-1"), ("Figura_A-2_boxplot_z_maximo.png", "Figura A-2")]:
    print(n); display(Image(f"revision/figuras/{f}", width=900))
